In [ ]:
from src.calibrateBPRData import CalibrationCoefficients as CCD
import pandas as pd
pd.set_option('display.precision', 15)


In [ ]:
# define the SNs for BPR logger, paroScientific SN, platinum SN
CCD = CCD()
paroSensorID= 93996 # based on logfile from recovery
loggerID = 0xB9 # paroscientific
TempSensorID = 0x98 # platinum

### read in the raw data file

In [ ]:
df = pd.read_csv('out/NCHR_rawHex_20250101_20250201.csv',index_col='times')
df.head()

In [ ]:
df.index = pd.DatetimeIndex(df.index)
df.index

### define the output dictionary and get the calibration coefficients stored in platinum.txt and parosci.txt

In [ ]:
Out={'DMASTime':[],'Time':[],'T_Housing':[],'P_SF':[],'T_SF':[]}
Out['DMASTime']=df.index
CP_SF=CCD.getParoCoeffs(paroSensorID) # sensor ID for pressure gaugeSF
CT_Ti=CCD.getPlatinumCoeffs(TempSensorID) # sensor ID for thermistor 

In [ ]:
# Paroscienticif coefficients
CP_SF

In [ ]:
# platinum Coefficients
CT_Ti

### Read each hexLine and convert to ASCII and separate the blocks for each sensor

In [ ]:
import re

Raw=[]
for line in df['readings'].values:
    try:
                        # Minimum sample with time, id, Tint, 1 Paros, and 00 has 26 characters
                        hexBlock=re.search(r'[\dA-Fa-f]{26,}',line)
                        #print(hexBlock.group(0))
                        x=re.findall(r'[\dA-Fa-f]{8}',hexBlock.group(0))
                        x[1]=x[1][2:]
                        t=CCD.calibratePPCTime(int(x[0],16)).strftime('%Y-%m-%d %H:%M:%S')
                        if not x[2] == 'FFFFFFFF': # error in some data
                            xData=[int(xi,16) for xi in x[1:]]
                        else:
                            xData=[int(xi,16) for xi in x[1:]]
                            xData[1]=0
                        Raw.append([t,xData])
                        #                    Data=CCD.calibrateData(xData)
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
#optional write it into a Dataframe to get the frequency periods or step to calibration cell below directly
df2 = pd.DataFrame(Raw)

In [ ]:
df2.head()

In [ ]:
def getFrequencyPeriods(xFT,xFP):
    X=(((xFT+4294967296)*4.656612873e-9)/4) # Temperature period in micro-sec
    T = (xFP+4294967296)*4.656612873e-9 # Pressure period in micro-sec
    return(X,T)

In [ ]:
# define the pressure and temperature period variables
X=[]
T=[]

In [ ]:
# Attention to the order of ascii blocks in the input data
# position [0] = housing temp
# position [1] = seawater temp (paro temp)
# postition [2] = seater pressure

xFTs = [df2[1][i][1] for i,j in enumerate(df2[1])]
xFPs = [df2[1][i][2] for i,j in enumerate(df2[1])]

In [ ]:

for i,j in enumerate(xFTs):
    X.append(getFrequencyPeriods(xFTs[i],xFPs[i])[0])
    T.append(getFrequencyPeriods(xFTs[i],xFPs[i])[1])

In [ ]:
# test the formula
import numpy as np
for i,j in enumerate(xFTs):
    U=((xFTs[0]+4294967296)*4.656612873e-9/4)-CP_SF['U0']
    Temp = CP_SF['Y1']*U+CP_SF['Y2']*np.power(U,2)
    print(Temp)

In [ ]:
df2['Pperiod']=T
df2['Tperiod']=X

In [ ]:
df2.head()

### calibrate with function in calibrateBPRData.py and using the above dictionary output "Raw"


In [ ]:
for i,iRaw in enumerate(Raw[0:3]):
    if len(iRaw[1]) ==3:
        Out['Time'].append(iRaw[0])
        Out['T_Housing'].append(CCD.calibratePlatinum(iRaw[1][0],Coeffs=CT_Ti))
        
        Out['T_SF'].append(CCD.calibrateParoT(iRaw[1][1],Coeffs=CP_SF))
        Out['P_SF'].append(CCD.calibrateParoP(iRaw[1][2],Coeffs=CP_SF,Temp=Out['T_SF'][i]))


In [ ]:
# calibrated values
Out